# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR² dataset using the `mlcroissant` library, referencing all dataset entities using their `@id` fields as specified in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and make a first inspection using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All references are made using official Croissant `@id` identifiers.

In [ ]:
# List all available record sets and their fields with their `@id`

record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")
record_set_ids = []

for rs in record_sets:
    print(f"RecordSet: {rs.id}")
    record_set_ids.append(rs.id)
    fields = list(rs.fields)
    print("Fields in this RecordSet:")
    for field in fields:
        print(f"  Field @id: {field.id}  (name: {field.name})")
    print('-'*40)

# As an example, print the first few record sets and fields
display_limit = 1
if len(record_sets) > display_limit:
    print(f"Showing the first {display_limit} record set(s).")

## 3. Data Extraction
Extract data from a selected record set as a DataFrame for analysis. All references to record-sets and fields use their `@id`s.

In [ ]:
# Choose a record set by its @id for analysis (using the first available record set)
main_record_set_id = record_set_ids[0]

# Load all records from this record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print(f"Fields (columns) in record set {main_record_set_id}:")
print(df.columns.tolist())
# Show a sample of the data
df.head()

## 4. Exploratory Data Analysis (EDA)
We will perform EDA with field references by `@id` as defined in the Croissant metadata. Please refer to section 2 output or the Croissant schema for valid `@id` values per numeric or grouping fields.

In [ ]:
# Example: Analyze a numeric field and group by another categorical field
# You can adjust these field IDs according to the dataset metadata; here we guess example IDs based on common MS datasets.

# Find a numeric field (e.g., percentage_coverage, molecular_weight, peptide_counts)
example_numeric_fields = [c for c in df.columns if 'coverage' in c or 'weight' in c or 'count' in c or 'abundance' in c or 'mw' in c]
print('Numeric field candidates:', example_numeric_fields)
numeric_field_id = example_numeric_fields[0] if example_numeric_fields else df.select_dtypes('number').columns[0]

# Find a grouping field (e.g. protein description, accession, etc.)
example_group_fields = [c for c in df.columns if 'accession' in c or 'description' in c or 'sample' in c or 'id' in c]
group_field_id = example_group_fields[0] if example_group_fields else df.columns[0]

# Ensure the numeric field is float for analysis
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Drop records where our field is NaN
filtered_df = df[df[numeric_field_id].notnull()]

# Set a threshold: filter on values > median
threshold = filtered_df[numeric_field_id].median()
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} (first 5 records):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())

# Group by a categorical field (if present)
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped data by {group_field_id} (top 5 groups):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. All field references use their `@id` identifiers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Box plot grouped by a categorical field (if group_field_id is valid)
if group_field_id in df.columns and df[group_field_id].nunique() < 30:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze the Mass Spectrometry Analysis of Extracellular Vesicles dataset using `mlcroissant`. All operations referenced fields and record sets using their official Croissant `@id` identifiers for full reproducibility.

- We verified the dataset structure and fields via metadata.
- We loaded and normalized protein-level data.
- We visualized key numeric fields (e.g. coverage, abundance, MW, etc.).

Feel free to further extend this notebook by referencing additional @id fields, joining across record sets, or applying advanced analysis methods!